# 🖼️ PyGame Catch-Up: Sprites & Images
> **Who is this for?** Students who missed Lesson 2. Work through top-to-bottom — run every code cell before moving on.

**By the end you will be able to:**
- Load a PNG image with `pygame.image.load()`
- Use `.convert_alpha()` and explain why it matters
- Draw (blit) images onto the screen in the right order
- Scale and rotate sprites with `pygame.transform`
- Load a background image and layer sprites on top
- Build a keyboard-controlled sprite explorer

> ⏱️ **Estimated time:** 60–90 minutes  
> 📁 **You need:** an `assets/` folder with at least `player.png` and `background.png`


---
## 🪟 Section 1 — Everything is a Surface

In PyGame, **every visible thing is a Surface** — the game window, every image you load, every shape you draw onto. Think of surfaces like sheets of paper you can stack on top of each other.

**Blitting** means copying one surface onto another — like stamping an ink block on paper.

```
screen  (the big surface — your window)
┌────────────────────────────────────┐
│  background (surface)              │
│  ┌──────────────────────────┐      │
│  │  player sprite (surface) │      │
│  └──────────────────────────┘      │
└────────────────────────────────────┘
```

**Order matters:** blit background first, then sprites on top. Later blits appear in front.


### 🧠 Check Your Understanding
Answer as comments:

In [ ]:
# 1. What is a Surface in PyGame?
#

# 2. What does 'blit' mean?
#

# 3. If you blit the player BEFORE the background, what happens?
#


---
## 📂 Section 2 — Loading Images

```python
# Basic pattern — always follow this order:
pygame.init()                                        # 1. init first
img = pygame.image.load('assets/player.png')        # 2. load
img = img.convert_alpha()                           # 3. convert
```

### Why `.convert_alpha()`?
- Converts the image to match the screen's pixel format
- Preserves PNG transparency (the invisible background stays invisible)
- Makes drawing 2–3× faster
- **Rule: always call it after loading any image with transparency**

Use `.convert()` (without alpha) for images with no transparency — slightly faster still.


### 🔍 Reading Exercise
Read the code, predict what happens at each `#?`, then run to check.

In [ ]:
import pygame
pygame.init()
screen = pygame.display.set_mode((400, 300))

# What happens if we swap these two lines? Try it!
import os
path = os.path.join('assets', 'player.png')

if os.path.exists(path):
    img = pygame.image.load(path).convert_alpha()
    print('Loaded!  Size:', img.get_width(), 'x', img.get_height(), 'pixels')
    print('Has per-pixel alpha:', img.get_flags() & pygame.SRCALPHA != 0)
else:
    print('assets/player.png not found — create the folder and add a PNG to test loading.')
    print('The rest of the notebook will still work with fallback shapes.')

pygame.quit()


### ✏️ Fill in the blanks

In [ ]:
# Complete each line — replace ??? with the correct call

# pygame.init() must be called before loading images
pygame.init()
screen = pygame.display.set_mode((800, 600))

# Load player.png from the assets folder, with transparency
player = pygame.image.???("assets/player.png").???()    # load & convert

# Load background.png — no transparency needed, use faster convert()
bg = pygame.image.load("assets/background.png").???()   # no alpha

# Get the image dimensions
width  = player.get_???()    # width in pixels
height = player.get_???()    # height in pixels
print(f'Player size: {width} x {height}')

pygame.quit()


---
## 🖨️ Section 3 — Blitting (Drawing Sprites)

```python
# surface.blit(image, position)
screen.blit(background, (0, 0))          # top-left corner
screen.blit(player,     (100, 200))       # player at (100, 200)
screen.blit(player,     player.get_rect(center=(400, 300)))  # centred
```

The position is the **top-left corner** of where the image will be placed.

### Blit order = draw order
Whatever you blit last appears on top. Think of it as painting — later brushstrokes cover earlier ones.


### ✏️ Blit Order Challenge
Number these lines 1–4 so the player appears in front of everything else:

In [ ]:
# Re-order these by changing the numbers in the comments
# Currently they're in the wrong order — the player gets painted over!

# Step ?: screen.blit(player_img,  (350, 400))   # player
# Step ?: screen.blit(cloud_img,   (200, 50))    # cloud
# Step ?: screen.blit(ground_img,  (0, 500))     # ground
# Step ?: screen.blit(sky_img,     (0, 0))       # sky background

# Write the correct order here:
# 1st: ___________
# 2nd: ___________
# 3rd: ___________
# 4th: ___________


---
## 🎮 Section 4 — Working Sprite Demo

Here is a full PyGame program that loads a sprite and lets you move it with arrow keys. If `assets/player.png` doesn't exist, it draws a coloured rectangle instead — the game always works.

Read through the comments carefully, then run it.


In [ ]:
import pygame, sys, os

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption('Sprite Demo')
clock = pygame.time.Clock()

# ── Load sprite (with fallback) ──────────────────────────────────────
PLAYER_W, PLAYER_H = 50, 70   # size we want regardless of source image
player_path = os.path.join('assets', 'player.png')

if os.path.exists(player_path):
    # Load and scale to our target size in one chain
    player_img = pygame.image.load(player_path).convert_alpha()
    player_img = pygame.transform.scale(player_img, (PLAYER_W, PLAYER_H))
    using_sprite = True
else:
    player_img  = None     # no file — will draw a rectangle instead
    using_sprite = False

# ── Load background (with fallback) ──────────────────────────────────
bg_path = os.path.join('assets', 'background.png')
if os.path.exists(bg_path):
    bg_img = pygame.image.load(bg_path).convert()
    bg_img = pygame.transform.scale(bg_img, (WIDTH, HEIGHT))  # fit to window
else:
    bg_img = None   # will fill with solid colour

# ── Starting position ─────────────────────────────────────────────────
player_x = WIDTH  // 2 - PLAYER_W // 2
player_y = HEIGHT // 2 - PLAYER_H // 2
SPEED = 5

# ── Colours (for fallback shapes) ────────────────────────────────────
CYAN  = (40, 200, 220)
DARK  = (20, 20, 35)

running = True
while running:

    # Events
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            running = False

    # Update — arrow key movement
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:  player_x -= SPEED
    if keys[pygame.K_RIGHT]: player_x += SPEED
    if keys[pygame.K_UP]:    player_y -= SPEED
    if keys[pygame.K_DOWN]:  player_y += SPEED

    # Clamp so sprite stays on screen
    player_x = max(0, min(player_x, WIDTH  - PLAYER_W))
    player_y = max(0, min(player_y, HEIGHT - PLAYER_H))

    # Draw — background first, then player on top
    if bg_img:
        screen.blit(bg_img, (0, 0))   # image background
    else:
        screen.fill(DARK)              # solid colour fallback

    if player_img:
        screen.blit(player_img, (player_x, player_y))       # sprite
    else:
        pygame.draw.rect(screen, CYAN,
                         (player_x, player_y, PLAYER_W, PLAYER_H))  # shape fallback

    pygame.display.flip()
    clock.tick(60)

pygame.quit()


### 🧠 Questions — answer as comments in the cell below
1. Why do we call `pygame.transform.scale()` after loading the image?
2. Why is `bg_img` loaded with `.convert()` but `player_img` with `.convert_alpha()`?
3. What does the clamp (`max(0, min(...))`) prevent?
4. What would happen if you blitted `player_img` before `bg_img`?


In [ ]:
# 1.
# 2.
# 3.
# 4.


---
## 🔄 Section 5 — pygame.transform

The `pygame.transform` module lets you resize, rotate, and mirror surfaces.

| Function | What it does |
|----------|--------------|
| `pygame.transform.scale(img, (w, h))` | Resize to exact pixel size |
| `pygame.transform.scale_by(img, factor)` | Scale by multiplier (2 = double) |
| `pygame.transform.rotate(img, angle)` | Rotate; positive = anti-clockwise |
| `pygame.transform.flip(img, x, y)` | Mirror horizontally/vertically |

⚠️ **Critical rule:** these return a **new** surface — they don't modify the original.

```python
# BAD — degrades quality every frame
player_img = pygame.transform.rotate(player_img, 3)    # ❌

# GOOD — always rotate from the original master
rotated = pygame.transform.rotate(player_img_orig, angle)   # ✅
screen.blit(rotated, (x, y))
```


### 🔍 Trace the transforms
For each line, predict the output dimensions. Starting image: 100 × 80 pixels.

In [ ]:
# A 100×80 image. What are the output dimensions after each transform?
# Write your predictions as comments, then run to check.

import pygame
pygame.init()
screen = pygame.display.set_mode((400, 300))

# Create a dummy 100×80 surface to test transforms
original = pygame.Surface((100, 80))
original.fill((255, 0, 0))

# Predict then check:
a = pygame.transform.scale(original, (200, 160))
print(f'scale to (200,160):     {a.get_width()} x {a.get_height()}')  # Prediction: ___

b = pygame.transform.scale_by(original, 0.5)
print(f'scale_by 0.5:           {b.get_width()} x {b.get_height()}')  # Prediction: ___

c = pygame.transform.rotate(original, 90)
print(f'rotate 90°:             {c.get_width()} x {c.get_height()}')  # Prediction: ___
# Hint: rotating by 90° swaps width and height!

d = pygame.transform.flip(original, True, False)
print(f'flip horizontal:        {d.get_width()} x {d.get_height()}')  # Prediction: ___

pygame.quit()


---
## 🕹️ Section 6 — Sprite Explorer (fill in the blanks)

Complete the sprite explorer program below. Every `???` needs to be replaced with correct code.
Use what you've learned in sections 1–5.

**Controls when it runs:**
- Arrow keys: move
- `+` / `-`: scale up / down
- `Q` / `E`: rotate left / right
- `ESC`: quit


In [ ]:
import pygame, sys, os

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption('Sprite Explorer')
clock = pygame.time.Clock()

# Load sprite (keep original for transforms)
player_path = os.path.join('assets', 'player.png')
if os.path.exists(player_path):
    player_orig = pygame.image.load(player_path).???()  # load with transparency
    player_orig = pygame.transform.scale(player_orig, (60, 80))
else:
    # Fallback: coloured surface
    player_orig = pygame.Surface((60, 80), pygame.SRCALPHA)
    player_orig.fill((40, 200, 220))

# Load background
bg_path = os.path.join('assets', 'background.png')
if os.path.exists(bg_path):
    bg = pygame.image.load(bg_path).convert()
    bg = pygame.transform.???(bg, (WIDTH, HEIGHT))  # scale to fill window
else:
    bg = None

# State
x, y   = WIDTH // 2, HEIGHT // 2
scale  = 1.0
angle  = 0
SPEED  = 5

running = True
while running:

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

    keys = pygame.key.get_pressed()

    # Movement
    if keys[pygame.K_LEFT]:  x -= SPEED
    if keys[pygame.K_RIGHT]: ??? += SPEED           # move right
    if keys[pygame.K_UP]:    y -= ???               # move up
    if keys[pygame.K_DOWN]:  y += SPEED

    # Scale
    if keys[pygame.K_EQUALS]: scale = min(scale + 0.02, 4.0)
    if keys[pygame.K_MINUS]:  scale = max(??? - 0.02, 0.2)  # shrink

    # Rotate
    if keys[pygame.K_q]: angle += 3               # rotate left
    if keys[pygame.K_e]: ??? -= 3                 # rotate right

    # Clamp to screen (using scaled dimensions)
    sw = int(player_orig.get_width()  * scale)
    sh = int(player_orig.get_height() * scale)
    x  = max(0, min(x, ??? - sw))                # clamp x
    y  = max(0, min(y, HEIGHT - ???))             # clamp y

    # Draw
    if bg:
        screen.blit(???, (0, 0))                  # draw background
    else:
        screen.fill((20, 20, 35))

    # Transform from ORIGINAL each frame (never chain!)
    display = pygame.transform.scale_by(???, scale)  # scale from original
    display = pygame.transform.rotate(???, angle)    # then rotate
    screen.???(display, (x, y))                      # blit to screen

    pygame.display.flip()
    clock.tick(60)

pygame.quit()


---
## 🐞 Section 7 — Bug Hunt

The program below has **four bugs**. Find each one, explain what's wrong in a comment, and write the corrected line.


In [ ]:
# ── Bug Hunt — find and fix all 4 bugs ──

import pygame, sys

# Bug 1: what's wrong with this loading sequence?
player = pygame.image.load('assets/player.png').convert_alpha()
pygame.init()    # <- should this be before or after load?

bg = pygame.image.load('assets/background.png').convert()
bg = pygame.transform.scale(bg, (800, 600))

x, y   = 400, 300
angle  = 0

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Bug 2: wrong blit order — player disappears!
    screen.blit(player, (x, y))    # player first
    screen.blit(bg, (0, 0))        # background on top  <- problem?

    # Bug 3: chaining transforms — quality degrades each frame
    player = pygame.transform.rotate(player, 2)   # <- what's wrong?
    screen.blit(player, (x, y))

    # Bug 4: wrong velocity update
    keys = pygame.key.get_pressed()
    if keys[pygame.K_RIGHT]:
        x = 5    # <- should this be = or +=?

    pygame.display.flip()
    clock.tick(60)

pygame.quit()


In [ ]:
# Write your fixes here:

# Bug 1 — what is wrong and how to fix it:
#

# Bug 2 — what is wrong and how to fix it:
#

# Bug 3 — what is wrong and how to fix it:
#

# Bug 4 — what is wrong and how to fix it:
#


---
## 🌟 Section 8 — Extensions

Finished? Add at least **two** of these to your sprite explorer from Section 6:

**1. Flip when moving left**
```python
if keys[pygame.K_LEFT]:
    display = pygame.transform.flip(display, True, False)  # mirror
```

**2. Show current values on screen**
```python
font = pygame.font.SysFont('consolas', 20)
info = font.render(f'scale: {scale:.2f}  angle: {angle}°', True, (255,255,255))
screen.blit(info, (10, 10))
```

**3. Trail effect** (blit a semi-transparent overlay instead of clearing)
```python
trail = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
trail.fill((0, 0, 0, 40))   # 40 out of 255 opacity
# Each frame, instead of screen.fill(), do:
screen.blit(trail, (0, 0))
```

**4. Second sprite** — add `enemy_x, enemy_y` and draw a second image that stays still in a corner

**5. Pulsing scale** — make the sprite gently grow and shrink:
```python
import math
pulse = 1.0 + 0.15 * math.sin(pygame.time.get_ticks() / 300)
display = pygame.transform.scale_by(player_orig, pulse)
```


In [ ]:
# Your extensions here:


---
## 📋 Quick Reference

### Loading
| Call | Use |
|------|-----|
| `pygame.image.load(path).convert_alpha()` | Image with transparency (PNG) |
| `pygame.image.load(path).convert()` | Image without transparency (faster) |
| `img.get_width()` / `img.get_height()` | Pixel dimensions |
| `img.get_rect()` | Rect matching image size |

### Blitting
| Call | Use |
|------|-----|
| `screen.blit(img, (x, y))` | Draw at top-left (x, y) |
| `screen.blit(img, img.get_rect(center=(cx, cy)))` | Draw centred at (cx, cy) |

### pygame.transform
| Call | Use |
|------|-----|
| `pygame.transform.scale(img, (w, h))` | Exact resize |
| `pygame.transform.scale_by(img, factor)` | Proportional resize |
| `pygame.transform.rotate(img, angle)` | Rotate (+ = anti-clockwise) |
| `pygame.transform.flip(img, x, y)` | Mirror horizontally / vertically |

### Golden Rules
1. `pygame.init()` before everything else
2. `.convert_alpha()` after loading transparent images
3. Blit **background first**, sprites on top
4. Always transform from the **original** image, never the previous result
5. Store untouched master as `img_orig`; derive display versions each frame


---
## ✅ Completion Checklist

Before submitting, check each box mentally:

- [ ] I can explain what a Surface is
- [ ] I can load a PNG with `.convert_alpha()` and explain why it's needed
- [ ] I know that blit order = draw order (background first)
- [ ] I understand why we always transform from the original image
- [ ] I identified and fixed all four bugs in Section 7
- [ ] My sprite explorer runs without errors
- [ ] I completed at least two extensions

**Great work — you're ready for Lesson 3: Keyboard & Mouse Input! 🎮**
